# 00 — Preparação da base

Obtém os arquivos da fonte oficial, confere a integridade da base e separa os conjuntos por ano.
Nenhuma estatística do conjunto de teste é explorada aqui: ele fica reservado para a avaliação final.

In [1]:
from pathlib import Path
import sys

# Raiz do projeto: a primeira pasta, a partir da atual, que contém src/dados.py.
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "dados.py").exists())
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

In [2]:
import json

import numpy as np
import pandas as pd

from src.dados import (ANOS_POR_CONJUNTO, COLUNAS, PASTA_ARTEFATOS, PASTA_PROCESSADO, PASTA_RAW,
                       SHA256_ARQUIVOS, URL_UCI, baixar_base, ler_arquivos_anuais, sha256)

## Origem e carregamento dos dados

**Gas Turbine CO and NOx Emission Data Set** — UCI Machine Learning Repository, 2019.
DOI [10.24432/C5WC95](https://doi.org/10.24432/C5WC95), licença **CC BY 4.0** (exige atribuição).

São cinco arquivos anuais (`gt_2011.csv` a `gt_2015.csv`) com leituras horárias de 11 sensores de
uma turbina a gás. Se os arquivos não estiverem em `data/raw/`, eles são baixados da UCI; em seguida,
o SHA-256 de cada arquivo é comparado com o do arquivo publicado.

In [3]:
baixados = baixar_base(PASTA_RAW)
print("Fonte:", URL_UCI)
print("Arquivos baixados agora:", baixados or "nenhum (já estavam em data/raw)")
for nome in SHA256_ARQUIVOS:
    print(f"{nome}  SHA-256 conferido  {sha256(PASTA_RAW / nome)[:16]}…")

Fonte: https://archive.ics.uci.edu/static/public/551/gas+turbine+co+and+nox+emission+data+set.zip
Arquivos baixados agora: nenhum (já estavam em data/raw)
gt_2011.csv  SHA-256 conferido  d87ceef9aa59533c…
gt_2012.csv  SHA-256 conferido  be54b9d0e1a7de40…
gt_2013.csv  SHA-256 conferido  13c437bb440ec204…
gt_2014.csv  SHA-256 conferido  c2a03c92c9c3207a…
gt_2015.csv  SHA-256 conferido  9b08f35fde0d4b13…


In [4]:
base = ler_arquivos_anuais(PASTA_RAW)
base.head()

,AT,AP,AH,AFDP,GTEP,TIT,TAT,TEY,CDP,CO,NOX,ano
0,4.5878,1018.7,83.675,3.5758,23.979,1086.2,549.83,134.67,11.898,0.32663,81.952,2011
1,4.2932,1018.3,84.235,3.5709,23.951,1086.1,550.05,134.67,11.892,0.44784,82.377,2011
2,3.9045,1018.4,84.858,3.5828,23.990,1086.5,550.19,135.10,12.042,0.45144,83.776,2011
3,3.7436,1018.3,85.434,3.5808,23.911,1086.5,550.17,135.03,11.990,0.23107,82.505,2011
4,3.7516,1017.8,85.182,3.5781,23.917,1085.9,550.00,134.67,11.910,0.26747,82.028,2011


## Estrutura e qualidade da base

Conferências de dimensões, nomes, tipos, valores ausentes e registros duplicados. A coluna `ano`
não existe nos arquivos originais: ela vem do nome de cada arquivo e serve só para separar os conjuntos.

In [5]:
print("Dimensões:", base.shape)
print("Colunas dos arquivos:", list(base.columns.drop("ano")))
print("Colunas esperadas   :", COLUNAS)
pd.DataFrame({"tipo": base[COLUNAS].dtypes.astype(str),
              "ausentes": base[COLUNAS].isna().sum()})

Dimensões: (36733, 12)
Colunas dos arquivos: ['AT', 'AP', 'AH', 'AFDP', 'GTEP', 'TIT', 'TAT', 'TEY', 'CDP', 'CO', 'NOX']
Colunas esperadas   : ['AT', 'AP', 'AH', 'AFDP', 'GTEP', 'TIT', 'TAT', 'TEY', 'CDP', 'CO', 'NOX']


,tipo,ausentes
AT,float64,0
AP,float64,0
AH,float64,0
AFDP,float64,0
GTEP,float64,0
TIT,float64,0
TAT,float64,0
TEY,float64,0
CDP,float64,0
CO,float64,0


In [6]:
qualidade = pd.DataFrame({
    "registros": base.groupby("ano").size(),
    "valores_ausentes": base.groupby("ano")[COLUNAS].apply(lambda g: int(g.isna().sum().sum())),
    "registros_duplicados_no_ano": base.groupby("ano")[COLUNAS].apply(lambda g: int(g.duplicated().sum())),
})
qualidade

,registros,valores_ausentes,registros_duplicados_no_ano
ano,,,
2011,7411,0,0
2012,7628,0,0
2013,7152,0,0
2014,7158,0,7
2015,7384,0,0


A base não tem valores ausentes: por isso as falhas de sensor serão **simuladas**, escondendo
leituras de registros completos. Os poucos registros repetidos são mantidos como estão, pois podem
ser horas com a mesma operação; nenhum registro é removido.

## Organização dos dados por ano

A divisão segue a orientação dos autores da base:

| Conjunto | Anos | Uso |
|---|---|---|
| Treino | 2011–2012 | aprendizado e ajuste do pré-processamento |
| Validação | 2013 | experimentos, comparação e seleção |
| Teste | 2014–2015 | avaliação final |

Cada conjunto é gravado em `data/processado/`. Nenhuma transformação é ajustada aqui.

In [7]:
PASTA_PROCESSADO.mkdir(parents=True, exist_ok=True)
for nome, anos in ANOS_POR_CONJUNTO.items():
    conjunto = base[base["ano"].isin(anos)][COLUNAS + ["ano"]]
    conjunto.to_csv(PASTA_PROCESSADO / f"{nome}.csv", index=False)
    print(f"{nome:10s} anos {anos}  ->  {len(conjunto):6d} registros  ({PASTA_PROCESSADO.name}/{nome}.csv)")

treino     anos (2011, 2012)  ->   15039 registros  (processado/treino.csv)


validacao  anos (2013,)  ->    7152 registros  (processado/validacao.csv)
teste      anos (2014, 2015)  ->   14542 registros  (processado/teste.csv)


## Validação da preparação

Verificações automáticas. Se alguma falhar, o notebook para com erro.

In [8]:
contagem_esperada = {2011: 7411, 2012: 7628, 2013: 7152, 2014: 7158, 2015: 7384}

assert list(base.columns.drop("ano")) == COLUNAS, "nomes ou ordem das colunas diferentes"
assert base.groupby("ano").size().to_dict() == contagem_esperada, "contagem por ano diferente"
assert len(base) == 36733
assert base[COLUNAS].notna().all().all(), "há valores ausentes"
assert all(np.issubdtype(t, np.number) for t in base[COLUNAS].dtypes), "há colunas não numéricas"
assert (base["CO"] >= 0).all(), "CO negativo impediria o log1p"

recarregados = {nome: pd.read_csv(PASTA_PROCESSADO / f"{nome}.csv") for nome in ANOS_POR_CONJUNTO}
anos_vistos = {nome: set(df["ano"]) for nome, df in recarregados.items()}
assert anos_vistos == {nome: set(anos) for nome, anos in ANOS_POR_CONJUNTO.items()}
assert sum(len(df) for df in recarregados.values()) == len(base)
assert anos_vistos["treino"].isdisjoint(anos_vistos["validacao"] | anos_vistos["teste"])
assert anos_vistos["validacao"].isdisjoint(anos_vistos["teste"])
for nome, df in recarregados.items():
    assert list(df.columns) == COLUNAS + ["ano"]
print("Todas as verificações passaram.")

Todas as verificações passaram.


In [9]:
manifesto = {
    "base": "Gas Turbine CO and NOx Emission Data Set (UCI 551)",
    "doi": "https://doi.org/10.24432/C5WC95",
    "licenca": "CC BY 4.0",
    "sha256": {nome: sha256(PASTA_RAW / nome) for nome in SHA256_ARQUIVOS},
    "registros_por_ano": {str(a): int(n) for a, n in base.groupby("ano").size().items()},
    "registros_por_conjunto": {nome: len(df) for nome, df in recarregados.items()},
    "anos_por_conjunto": {nome: list(anos) for nome, anos in ANOS_POR_CONJUNTO.items()},
    "colunas": COLUNAS,
}
PASTA_ARTEFATOS.mkdir(parents=True, exist_ok=True)
(PASTA_ARTEFATOS / "manifesto_dados.json").write_text(json.dumps(manifesto, indent=2, ensure_ascii=False),
                                                      encoding="utf-8")
manifesto["registros_por_conjunto"]

{'treino': 15039, 'validacao': 7152, 'teste': 14542}

**Resultado:** `data/processado/treino.csv`, `validacao.csv` e `teste.csv`, além de
`artefatos/manifesto_dados.json`. Os outros notebooks carregam os conjuntos por
`src.dados.carregar_conjunto`, que só abre o teste com `liberar_teste=True`.